# QEC 513 Delay Benchmark

Standalone fidelity benchmark for the five-qubit code. The notebook compares a noise-free Aer reference to optional IBM hardware delay sweeps and saves QEC benchmark JSON files under `results/qec513/`.


In [ ]:
from datetime import datetime
import uuid
import json
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
from qiskit.transpiler import generate_preset_pass_manager
from qiskit_aer.primitives import SamplerV2 as AerSampler
from qiskit_ibm_runtime import QiskitRuntimeService, SamplerV2 as Sampler

from broadcasting import HardwareBackend, qec_513_delay_benchmark_circuit
from broadcasting.analysis import delay_axis
from broadcasting.provenance import software_provenance, circuit_provenance, calibration_provenance
from broadcasting.results import write_run_json
from scripts.figure_sources import load_source, source_manifest
from broadcasting.plotting import save_figure

%matplotlib inline
plt.rcParams.update({"figure.dpi": 120})


## Configuration


In [ ]:
USE_QEC = True
RUN_HARDWARE = False
SAVE_FIGURE = True

seed = 42
rng = np.random.default_rng(seed)
theta = float(rng.uniform(0, np.pi))
phi = float(rng.uniform(0, 2 * np.pi))
tau_values = np.linspace(0, 6000, 21).astype(int).tolist()
shots = 8192

IBM_PROFILE = "mprest1"
IBM_BACKEND = "ibm_kingston"
OPTIMIZATION_LEVEL = 0

QEC_RESULTS_DIR = Path("results/qec513")
QEC_RESULTS_DIR.mkdir(parents=True, exist_ok=True)
FIGURE_DIR = Path("figures")

print(f"USE_QEC={USE_QEC}  theta={theta:.4f}  phi={phi:.4f}")
print(f"tau values: {tau_values[0]}..{tau_values[-1]} dt ({len(tau_values)} points)")


## Build Circuit


In [ ]:
qc, tau_param, fid_reg = qec_513_delay_benchmark_circuit(
    theta,
    phi,
    use_qec=USE_QEC,
)
print(f"Circuit depth: {qc.depth()}")
print(f"Qubits: {qc.num_qubits}")
qc.draw("mpl", fold=120)


## Noise-Free Reference


In [ ]:
def fidelity_from_pub(pub_result, register_name):
    counts = getattr(pub_result.data, register_name).get_counts()
    total = sum(counts.values())
    return sum(v for bitstring, v in counts.items() if bitstring[-1] == "0") / total

bound_circuits = [qc.assign_parameters({tau_param: int(t)}) for t in tau_values]
aer_result = AerSampler().run([(c,) for c in bound_circuits], shots=shots).result()
ideal_fidelities = [fidelity_from_pub(pub, fid_reg) for pub in aer_result]
print(f"Noise-free mean fidelity: {np.mean(ideal_fidelities):.4f}")


## Optional Hardware Sweep


In [ ]:
backend_fidelities = []
backend_counts = []
qec_outfile = None
backend_dt = None

if RUN_HARDWARE:
    tau_values = HardwareBackend._validate_taus(tau_values)
    service = QiskitRuntimeService(name=IBM_PROFILE)
    backend = service.backend(name=IBM_BACKEND)
    backend_dt = float(backend.dt)
    print(f"Backend: {backend.name}, dt={backend_dt} seconds")

    pass_manager = generate_preset_pass_manager(
        backend=backend, optimization_level=OPTIMIZATION_LEVEL, seed_transpiler=seed,
    )
    isa_circuit = pass_manager.run(qc)
    isa_circuits = [isa_circuit.assign_parameters({tau_param: int(t)}) for t in tau_values]
    HardwareBackend._validate_target(isa_circuits, backend)
    # Capture configuration and cached target provenance before submitting.
    execution_provenance = {
        "dt": backend_dt,
        "software": software_provenance(),
        "input_circuit": circuit_provenance(qc),
        "compiled_templates": [circuit_provenance(isa_circuit, backend.target)],
        "compiled_pubs": [{"template_index": 0, "bindings": {tau_param.name: int(t)}, "tau_dt": int(t), "tau_index": i} for i, t in enumerate(tau_values)],
        "calibration": calibration_provenance(backend, isa_circuits),
        "delay_order": [int(t) for t in tau_values],
        "seed_transpiler": seed,
    }
    job = Sampler(mode=backend).run([(c,) for c in isa_circuits], shots=shots)
    print(f"Job ID: {job.job_id()}")
    hardware_result = job.result()
    aligned_shots = []
    for pub in hardware_result:
        counts = getattr(pub.data, fid_reg).get_counts()
        total = sum(counts.values())
        backend_fidelities.append(sum(v for bits, v in counts.items() if bits[-1] == "0") / total)
        backend_counts.append(dict(counts))
        aligned_shots.append(HardwareBackend._aligned_shots(pub.data))

    timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
    qec_outfile = QEC_RESULTS_DIR / f"qec513_delay_sweep_{timestamp}_{uuid.uuid4().hex}.json"
    write_run_json({
        "timestamp": datetime.now().isoformat(), "experiment": "qec_513_delay_sweep",
        "job_id": job.job_id(), "optimization_level": OPTIMIZATION_LEVEL,
        "backend": backend.name, "shots": shots, "seed": seed, "use_qec": USE_QEC,
        "state_prep": {"theta": theta, "phi": phi},
        "tau_values": [int(t) for t in tau_values],
        "ideal_fidelities": [float(f) for f in ideal_fidelities],
        "backend_fidelities": [float(f) for f in backend_fidelities],
        "backend_counts": backend_counts,
        "metadata": {**execution_provenance, "aligned_shots": aligned_shots},
    }, qec_outfile)
    print(f"Saved hardware sweep to {qec_outfile}")
else:
    print("Hardware collection is disabled (RUN_HARDWARE=False).")


## Plot Current Sweep


In [ ]:
scale, unit = delay_axis({"metadata": {"dt": backend_dt}})
x_delay = scale * np.asarray(tau_values, dtype=float)
fig, ax = plt.subplots(figsize=(8, 5))
ax.plot(x_delay, ideal_fidelities, "o-", color="tab:green", label="Noise-free")
if backend_fidelities:
    ax.plot(
        x_delay,
        backend_fidelities,
        "s-",
        color="tab:red",
        label=f"{IBM_BACKEND} opt={OPTIMIZATION_LEVEL}",
    )
ax.axhline(0.5, color="gray", linestyle="--", alpha=0.5, label="Random (0.5)")
ax.set_xlabel(f"Delay time ({unit})")
ax.set_ylabel("Fidelity")
ax.set_ylim(0, 1.05)
ax.grid(alpha=0.25)
ax.legend()
plt.tight_layout()

if SAVE_FIGURE:
    suffix = "qec" if USE_QEC else "no_qec"
    figure_path = FIGURE_DIR / f"qec513_delay_sweep_{suffix}.png"
    save_figure(fig, figure_path)
    print(f"Saved title-free figure to {figure_path}")
plt.show()


## Compare Saved Memory Sweeps

Historical `use_qec` and `dt` overrides are explicit, source-backed entries in `figures/sources.json`; they never change raw records. Unattributed fields remain unknown. Duplicate job IDs are excluded. Runs without a documented conversion are shown in native dt in a separate panel, so units cannot be mixed silently.


In [ ]:
manifest = source_manifest()
saved = []
seen_jobs = set()
for path in sorted(QEC_RESULTS_DIR.glob("qec513_delay_sweep_*.json")):
    relative = path.as_posix()
    record = load_source(relative, manifest=manifest) if relative in manifest["runs"] else json.loads(path.read_text())
    job_id = record.get("job_id")
    if job_id and job_id in seen_jobs:
        print(f"Skipping duplicate job: {path.name}")
        continue
    seen_jobs.add(job_id)
    saved.append(record)
    print(f"{path.name}: {record.get('backend')} opt={record.get('optimization_level')} "
          f"use_qec={record.get('use_qec')}; provenance={record.get('historical_provenance', {})}")

if not saved:
    print("No saved memory sweeps found.")
else:
    units = sorted({delay_axis(record)[1] for record in saved})
    fig, axes = plt.subplots(1, len(units), figsize=(7 * len(units), 4.5), squeeze=False)
    for unit, ax in zip(units, axes[0]):
        for record in saved:
            scale, record_unit = delay_axis(record)
            if record_unit != unit:
                continue
            qec = record.get("use_qec")
            encoding = "unknown encoding" if qec is None else ("encoded" if qec else "bare")
            label = f"{record['backend']} {encoding} opt={record['optimization_level']} {record['timestamp'][:10]}"
            ax.plot(scale * np.asarray(record["tau_values"]), record["backend_fidelities"], label=label)
        ax.axhline(0.5, color="gray", linestyle="--", alpha=0.5)
        ax.set(xlabel=f"Delay ({unit})", ylabel="Recovered-state fidelity", ylim=(0, 1.05))
        ax.legend(fontsize=7)
        ax.grid(alpha=0.25)
    plt.tight_layout()
    if SAVE_FIGURE:
        save_figure(fig, FIGURE_DIR / "qec513_all_saved_sweeps.png")
    plt.show()
